# 第10回 自然言語処理：値とコードを一段ずつ読む

文字列の候補を選び、語を数え、ベクトルを比べ、予測に合わせて更新するまでを人工データで追います。

**開始：** 既存venvのカーネルを選び、初回は変更せず「カーネル再起動→すべて実行」。次に一節ずつ、予想してからShift+Enterで実行します。同梱Pythonのimportは不要で、外部通信も行いません。

**読み方：** 関数の定義は計算方法に名前を付けるセルです。続く短い実行セルが入力を渡します。表・図の表示コードは折り畳み指定です。表示は丸めますが計算には元の値を使います。

**完了の目印：** 研究＋生活、がの次の語の確率の和1、がのIDF=0、音のみなら雨・言語を加えると飴。


In [ ]:
import sys
print('使用中のPython:',sys.executable)
print('版:',sys.version.split()[0])


In [ ]:
import argparse


In [ ]:
import copy


In [ ]:
import json


In [ ]:
import math


In [ ]:
CORPUS = [['猫', 'が', '眠る'], ['犬', 'が', '眠る'],
          ['猫', 'が', '走る'], ['犬', 'が', '走る']]


In [ ]:
LEXICON = [('猫', '名詞', 1), ('犬', '名詞', 1), ('が', '助詞', 1),
           ('眠る', '動詞', 1), ('走る', '動詞', 1), ('研究', '名詞', 1),
           ('研究生', '名詞', 1), ('生活', '名詞', 1), ('活', '名詞', 5)]


In [ ]:
BOS, EOS = '<s>', '</s>'


### 図表の準備（変更不要）

表示用の補助コードです。HTML/SVGを覚える必要はありません。折り畳み表示はNotebook環境によって異なります。


In [ ]:
from html import escape


In [ ]:
from IPython.display import HTML, display


In [ ]:
def learning_format(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (list, tuple)):
        return '['+', '.join(learning_format(item) for item in value)+']'
    if isinstance(value, dict):
        return '{'+', '.join(str(key)+': '+learning_format(item) for key,item in value.items())+'}'
    return str(value)


In [ ]:
def learning_table(title, columns, rows):
    header = ''.join('<th>'+escape(str(c))+'</th>' for c in columns)
    body = ''
    for row in rows:
        body += '<tr>'+''.join('<td>'+escape(learning_format(x))+'</td>' for x in row)+'</tr>'
    return '<div class="scroll"><table><caption>'+escape(title)+'</caption><tr>'+header+'</tr>'+body+'</table></div>'


In [ ]:
def learning_grid(title, matrix):
    rows = [(i,*row) for i,row in enumerate(matrix)]
    return learning_table(title,['行↓ 列→',*range(len(matrix[0]))],rows)


In [ ]:
def learning_plot(title, points):
    # 実計算の値だけを座標へ変換。点の間の値は補間して表示しない。
    xs, ys = [p[0] for p in points], [p[1] for p in points]
    xmin,xmax,ymin,ymax = min(xs),max(xs),min(0,min(ys)),max(ys)
    sx = lambda x: 100+(x-xmin)/max(1e-12,xmax-xmin)*500
    sy = lambda y: 240-(y-ymin)/max(1e-12,ymax-ymin)*190
    marks = ''
    for value in (ymin,(ymin+ymax)/2,ymax):
        marks += f'<path d="M100 {sy(value)} H600" stroke="#aaa"/><text x="0" y="{sy(value)+5}">{value:.4g}</text>'
    for x,y in points:
        marks += f'<circle cx="{sx(x)}" cy="{sy(y)}" r="4" fill="#155e75"><title>{x}: {y}</title></circle>'
    marks += f'<text x="100" y="275">{xmin}</text><text x="580" y="275">{xmax}</text>'
    return '<svg viewBox="0 0 700 310" role="img" aria-label="'+escape(title)+'"><text x="100" y="22">'+escape(title)+'</text>'+marks+'</svg>'


In [ ]:
def learning_show(report, reset=None):
    style = '<style>.learning{font:17px/1.65 sans-serif;color:#172b42;background:white;padding:12px}.learning .scroll{overflow-x:auto}.learning table{border-collapse:collapse;margin:14px 0}.learning td,.learning th{border:1px solid #8394a5;padding:6px 12px;white-space:normal}.learning caption{text-align:left;font-weight:bold}.learning svg{max-width:100%;height:auto}.learning .boards{display:flex;gap:20px;flex-wrap:wrap}.learning .result{border-left:4px solid #155e75;padding:10px}</style>'
    content = '<h3>'+escape(report['title'])+'</h3><p>入力：'+escape(learning_format(report['input']))+'</p>'
    content += learning_table('使う値 → 操作 → 結果',report['columns'],report['rows'])
    for title,columns,rows in report.get('tables',[]):
        content += '<details><summary>詳しい計算：'+escape(title)+'</summary>'+learning_table(title,columns,rows)+'</details>'
    content += '<div class="boards">'+''.join(learning_grid(title,values) for title,values in report.get('matrices',[]))+'</div>'
    for title,points in report.get('plots',[]):
        content += learning_plot(title,points)
    for label in ('result','comparison','explanation','reset'):
        name={'result':'結果','comparison':'一つ変えた比較','explanation':'読み取り方','reset':'戻す'}[label]
        content += '<p class="result">'+name+'：'+escape(reset if label=='reset' and reset is not None else learning_format(report[label]))+'</p>'
    display(HTML(style+'<div class="learning">'+content+'</div>'))


## 1. 切り分け候補と合計コスト

**今いる段階：** 文字位置→候補→経路コスト

**予想：** 活のコストを下げると経路は変わるか

**操作：** example_costを5から0へ変更し、同じセルを実行します。

**結果の目印：** 研究＋生活から研究生＋活へ変わる

**戻す：** example_cost=5に戻して再実行します。

**説明する：** 単語の区切りを何で決めたか


### 計算の定義：`char_split`

字種の変わり目で区切る比較用処理。品詞は分からない。


In [ ]:
def char_split(text):
    """字種の変わり目で区切る比較用処理。品詞は分からない。"""
    if not isinstance(text, str) or len(text) > 80:
        raise ValueError('入力は80文字以内')

    def kind(c):
        if 'ぁ' <= c <= 'ゖ':
            return 'ひらがな'
        if 'ァ' <= c <= 'ヺ' or c == 'ー':
            return 'カタカナ'
        if '一' <= c <= '鿿':
            return '漢字'
        return 'その他'
    groups = []
    for c in text:
        if groups and kind(groups[-1][-1]) == kind(c):
            groups[-1] += c
        else:
            groups.append(c)
    return groups


### 計算の定義：`finite`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def finite(value, low, high, label):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or not low <= value <= high:
        raise ValueError(f'{label}: {low}〜{high}の有限数を指定')
    return value


### 計算の定義：`analyze`

先頭からのDP。接続コストなし。未知文字も常に候補に含める。


In [ ]:
def analyze(text, lexicon=LEXICON):
    """先頭からのDP。接続コストなし。未知文字も常に候補に含める。"""
    char_split(text)
    if not isinstance(lexicon, list) or not 1 <= len(lexicon) <= 40:
        raise ValueError('辞書は1〜40項目')
    for word, pos, cost in lexicon:
        if not isinstance(word, str) or not word or (not isinstance(pos, str)) or (not pos):
            raise ValueError('辞書の表層形・品詞は非空文字列')
        finite(cost, 0, 20, '辞書コスト')
    best = [None] * (len(text) + 1)
    best[0] = (0.0, (), [])
    edges = []
    for start in range(len(text)):
        for order, (word, pos, cost) in enumerate([*lexicon, (text[start], '未知', 10)]):
            if not text.startswith(word, start):
                continue
            end = start + len(word)
            edges.append({'start': start, 'end': end, 'word': word, 'pos': pos, 'cost': cost})
            old_cost, old_order, old_tokens = best[start]
            candidate = (old_cost + cost, old_order + (order,), old_tokens + [[word, pos, cost]])
            if best[end] is None or candidate[:2] < best[end][:2]:
                best[end] = candidate
    return {'tokens': best[-1][2], 'cost': best[-1][0], 'edges': edges}


### 計算の定義：`lesson_morph`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_morph(cost=5):
    lexicon = copy.deepcopy(LEXICON)
    lexicon[-1] = ('活','名詞',cost)
    result = analyze('研究生活',lexicon)
    rows = [(edge['start'],edge['end'],edge['word'],edge['pos'],edge['cost']) for edge in result['edges']]
    chosen, cumulative = [], 0
    for word,pos,value in result['tokens']:
        cumulative += value
        chosen.append((word,pos,value,cumulative))
    return {'title':'切り分け：候補のコストから経路を選ぶ','question':'研究＋生活と研究生＋活のどちらを選ぶか',
            'input':f'研究生活、活のコスト={cost}','columns':['開始文字','終了文字','候補','品詞','コスト'],'rows':rows,
            'tables':[('選択された経路を順に足す',['語','品詞','コスト','ここまでの合計'],chosen)],
            'result':f"選択={result['tokens']}、合計={result['cost']}",
            'comparison':f"元の辞書では{analyze('研究生活')['tokens']}",
            'explanation':'未知の1文字もコスト10の候補。合計が小さい経路を選び、同点は辞書の順。字種で区切るだけでは品詞は分からない。',
            'reset':'cost=5で再実行。元の辞書は変更しない。'}


In [ ]:
example_cost = 5
program_observation_morph = lesson_morph(example_cost)
learning_show(program_observation_morph, reset='example_cost=5に戻して再実行します。')


## 2. 回数を数えて確率にする

**今いる段階：** 出現回数→分母→確率

**予想：** 未観測の語へどう値を割り当てるか

**操作：** example_alphaを1から0へ変更し、同じセルを実行します。

**結果の目印：** α=0では未観測の語が0、確率の和は1

**戻す：** example_alpha=1に戻して再実行します。

**説明する：** 回数と分母の対応


### 計算の定義：`validate_corpus`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def validate_corpus(corpus):
    if not isinstance(corpus, list) or not 1 <= len(corpus) <= 20:
        raise ValueError('文書数は1〜20')
    for sentence in corpus:
        if not isinstance(sentence, list) or not 1 <= len(sentence) <= 20:
            raise ValueError('各文書の語数は1〜20（空文書不可）')
        if any(not isinstance(w, str) or not w or len(w) > 20 or any(c.isspace() for c in w) or w in (BOS, EOS) for w in sentence):
            raise ValueError('語は空白・予約記号を含まない1〜20文字')
    return list(dict.fromkeys(w for s in corpus for w in s))


### 計算の定義：`bigram`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def bigram(corpus=CORPUS, context='が', alpha=1):
    vocab = validate_corpus(corpus) + [EOS]
    finite(alpha, 0, 5, 'α')
    if context not in [BOS, *vocab] or context == EOS:
        raise ValueError('文脈は文頭またはコーパス内の語（文末以外）')
    counts = [0] * len(vocab)
    for sentence in corpus:
        seq = [BOS, *sentence, EOS]
        for before, after in zip(seq, seq[1:]):
            if before == context: counts[vocab.index(after)] += 1
    total = sum(counts)
    denominator = total + alpha * len(vocab)
    return {'vocab': vocab, 'context': context, 'counts': counts, 'total': total,
            'denominator': denominator, 'probabilities': [((n + alpha) / denominator) for n in counts]}


### 計算の定義：`lesson_bigram`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_bigram(alpha=1):
    result = bigram(alpha=alpha)
    rows = [(word,count,count+alpha,result['denominator'],probability) for word,count,probability in zip(result['vocab'],result['counts'],result['probabilities'])]
    return {'title':'出現回数から次の語の確率へ','question':'一度も出なかった語の確率はどうなるか',
            'input':f'4文書、直前の語=が、加算量α={alpha}','columns':['次の語','回数','回数＋α','全回数＋α×語彙数','割った確率'],'rows':rows,
            'result':f"分母={result['denominator']}、確率の合計={sum(result['probabilities'])}",
            'comparison':f"α=0の確率={bigram(alpha=0)['probabilities']}",
            'explanation':'がの次だけを数える。語彙には文末記号も含む。加算平滑化は観測されなかった語にも値を配る。',
            'reset':'alpha=1で再実行。固定コーパスは変更しない。'}


In [ ]:
example_alpha = 1
program_observation_bigram = lesson_bigram(example_alpha)
learning_show(program_observation_bigram, reset='example_alpha=1に戻して再実行します。')


## 3. 文書の特徴量を作る

**今いる段階：** 回数→TF→DF・IDF→積

**予想：** 頻出語は必ず強い特徴か

**操作：** example_documentを0から1へ変更し、同じセルを実行します。

**結果の目印：** がのDF=4、IDF=0

**戻す：** example_document=0に戻して再実行します。

**説明する：** TFとIDFの分母の違い


### 計算の定義：`tfidf`

語を数える→文書内比率TF→出現文書数DF→IDF→掛け算。


In [ ]:
def tfidf(corpus=CORPUS):
    """語を数える→文書内比率TF→出現文書数DF→IDF→掛け算。"""
    vocab = validate_corpus(corpus)
    counts, df, idf, tf, scores = [], [], [], [], []
    for sentence in corpus:
        counts.append([sentence.count(word) for word in vocab])
    for word in vocab:
        document_count = sum(word in sentence for sentence in corpus)
        df.append(document_count)
        idf.append(math.log(len(corpus) / document_count))
    for index, row in enumerate(counts):
        frequencies = []
        for count in row:
            frequencies.append(count / len(corpus[index]))
        tf.append(frequencies)
        scores.append([value * idf[j] for j,value in enumerate(frequencies)])
    return {'vocab': vocab, 'counts': counts, 'df': df, 'idf': idf, 'tf': tf, 'scores': scores}


### 計算の定義：`lesson_tfidf`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_tfidf(document=0):
    result = tfidf()
    rows = [(word,result['counts'][document][j],len(CORPUS[document]),result['tf'][document][j],result['df'][j],result['idf'][j],result['scores'][document][j]) for j,word in enumerate(result['vocab'])]
    other = (document+1)%len(CORPUS)
    return {'title':'語の回数→TF・IDF→文書の特徴量','question':'全部の文書にある「が」は強い特徴になるか',
            'input':f'文書{document}: {CORPUS[document]}、全4文書','columns':['語','文書内回数','文書語数','TF','DF','log(4/DF)','TF×IDF'],'rows':rows,
            'matrices':[('文書×語の回数',result['counts']),('文書×語のTF-IDF',result['scores'])],
            'result':result['scores'][document], 'comparison':f"隣の文書{other}の特徴量={result['scores'][other]}",
            'explanation':'TFの分母はその文書の語数、IDFは全体の文書数と出現文書数を使う。「が」はDF=4でIDF=0。',
            'reset':'document=0で再実行。語彙の並びは表の順。'}


In [ ]:
example_document = 0
program_observation_tfidf = lesson_tfidf(example_document)
learning_show(program_observation_tfidf, reset='example_document=0に戻して再実行します。')


## 4. ベクトルの類似度を読む

**今いる段階：** 周辺語→回数ベクトル→類似度

**予想：** 窓幅で何を数える対象が変わるか

**操作：** example_windowを1から2へ変更し、同じセルを実行します。

**結果の目印：** 共起行列と内積・長さを確認

**戻す：** example_window=1に戻して再実行します。

**説明する：** 共起が似ることの限界


### 計算の定義：`integer`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def integer(value, low, high, label):
    finite(value, low, high, label)
    if int(value) != value:
        raise ValueError(f'{label}: 整数を指定')
    return int(value)


### 計算の定義：`representations`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def representations(corpus=CORPUS, window=1):
    vocab = validate_corpus(corpus)
    window = integer(window, 1, 3, '窓幅')
    counts = [[0] * len(vocab) for _ in vocab]
    for s in corpus:
        for i, word in enumerate(s):
            for j in range(max(0, i - window), min(len(s), i + window + 1)):
                if j != i:
                    counts[vocab.index(word)][vocab.index(s[j])] += 1
    return {'vocab': vocab, 'onehot': [[int(i == j) for j in range(len(vocab))] for i in range(len(vocab))], 'cooccurrence': counts}


### 計算の定義：`cosine`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def cosine(a, b):
    if not a or len(a) != len(b):
        raise ValueError('同じ非ゼロ次元を指定')
    for x in [*a, *b]:
        finite(x, -1000000.0, 1000000.0, 'ベクトル要素')
    dot = sum((x * y for x, y in zip(a, b)))
    na = math.sqrt(sum((x * x for x in a)))
    nb = math.sqrt(sum((x * x for x in b)))
    return {'dot': dot, 'norm_a': na, 'norm_b': nb, 'cosine': dot / (na * nb) if na and nb else None}


### 計算の定義：`lesson_vectors`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_vectors(window=1):
    result = representations(window=window)
    a,b = result['cooccurrence'][0],result['cooccurrence'][3]
    similarity = cosine(a,b)
    rows = [(word,a[j],b[j],a[j]*b[j]) for j,word in enumerate(result['vocab'])]
    return {'title':'周辺の語を数え、ベクトルを比較する','question':'猫と犬の表記が違っても似たベクトルになるか',
            'input':f'中心語の左右{window}語を数える。猫と犬の行を比較。','columns':['周辺語','猫の回数','犬の回数','積'],'rows':rows,
            'matrices':[('語の順：'+str(result['vocab']),result['cooccurrence'])],
            'result':f'内積・長さ・cos類似度={similarity}',
            'comparison':f"窓幅2の猫の行={representations(window=2)['cooccurrence'][0]}",
            'explanation':'内積を二つの長さの積で割る。ゼロベクトルではcosは未定義。似た周辺語を持つことと同義語であることは同一ではない。',
            'reset':'window=1で再実行。'}


In [ ]:
example_window = 1
program_observation_vectors = lesson_vectors(example_window)
learning_show(program_observation_vectors, reset='example_window=1に戻して再実行します。')


## 5. ベクトルを一回更新する

**今いる段階：** 予測→差→勾配→更新

**予想：** 更新しない場合は何が残るか

**操作：** example_rateを0.1から0へ変更し、同じセルを実行します。

**結果の目印：** 学習率0では入力側・出力側とも不変

**戻す：** example_rate=0.1に戻して再実行します。

**説明する：** 予測出力と正解との差がどこへ使われたか


### 計算の定義：`RNG`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
class RNG:
    def __init__(self, seed=65535): self.state = integer(seed, 0, 2**32-1, 'seed')
    def random(self):
        self.state = (1664525 * self.state + 1013904223) % 2**32
        return self.state / 2**32


### 計算の定義：`initial_vectors`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def initial_vectors(size, seed=65535):
    size = integer(size, 1, 400, '語彙数'); rng = RNG(seed)
    return [[[rng.random() * .2 - .1 for _ in range(2)] for _ in range(size)] for _ in range(2)]


### 計算の定義：`objective`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def objective(inputs, outputs, center, target):
    scores = [sum((a * b for a, b in zip(inputs[center], row))) for row in outputs]
    maximum = max(scores)
    exps = [math.exp(v - maximum) for v in scores]
    total = sum(exps)
    probabilities = [v / total for v in exps]
    loss = maximum + math.log(total) - scores[target]
    errors = [p - int(i == target) for i, p in enumerate(probabilities)]
    grad_in = [sum((errors[j] * outputs[j][k] for j in range(len(outputs)))) for k in range(2)]
    grad_out = [[e * v for v in inputs[center]] for e in errors]
    return {'probabilities': probabilities, 'loss': loss, 'grad_in': grad_in, 'grad_out': grad_out}


### 計算の定義：`update`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def update(inputs, outputs, center, target, rate=0.1):
    finite(rate, 0, 1, '学習率')
    result = objective(inputs, outputs, center, target)
    ni, no = (copy.deepcopy(inputs), copy.deepcopy(outputs))
    ni[center] = [x - rate * g for x, g in zip(inputs[center], result['grad_in'])]
    no = [[x - rate * g for x, g in zip(row, grad)] for row, grad in zip(outputs, result['grad_out'])]
    return (ni, no, result)


### 計算の定義：`lesson_update`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_update(rate=0.1):
    vocab = validate_corpus(CORPUS)
    inputs, outputs = initial_vectors(len(vocab))
    new_inputs,new_outputs,result = update(inputs,outputs,0,1,rate)
    rows = [(word,result['probabilities'][j],int(j==1),result['probabilities'][j]-int(j==1)) for j,word in enumerate(vocab)]
    changes = [(j,inputs[0][j],result['grad_in'][j],rate,new_inputs[0][j]) for j in range(2)]
    return {'title':'中心語から周辺語を予測し、ベクトルを更新する','question':'学習率0では損失やベクトルが変わるか',
            'input':'中心語=猫、正解の周辺語=が、固定seedの2次元ベクトル','columns':['候補語','予測出力','正解なら1','差'],'rows':rows,
            'tables':[('猫の入力ベクトルの更新',['次元','更新前','勾配','学習率','更新後'],changes)],
            'matrices':[('更新前の出力側ベクトル',outputs),('更新後の出力側ベクトル',new_outputs)],
            'result':f"1組の損失 {result['loss']} → {objective(new_inputs,new_outputs,0,1)['loss']}",
            'comparison':f"学習率0では入力ベクトル不変={update(inputs,outputs,0,1,0)[0]==inputs}",
            'explanation':'差を使って入力側と出力側を更新する。両方の勾配は更新前の同じ値から求める。単語を数える表とは別の学習方法。',
            'reset':'rate=0.1で再実行。同じ初期ベクトルから1更新する。'}


In [ ]:
example_rate = 0.1
program_observation_update = lesson_update(example_rate)
learning_show(program_observation_update, reset='example_rate=0.1に戻して再実行します。')


## 6. 音と言語の候補スコア

**今いる段階：** 候補→二つのlog項→選択

**予想：** 文脈を外すと何を選ぶか

**操作：** example_weightを1から0へ変更し、同じセルを実行します。

**結果の目印：** λ=1は飴、λ=0は雨

**戻す：** example_weight=1に戻して再実行します。

**説明する：** 実際の音声認識精度の評価ではない理由


### 計算の定義：`recognize`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def recognize(acoustic=(0.6, 0.4), prior=(0.2, 0.8), weight=1):
    if len(acoustic) != 2 or len(prior) != 2:
        raise ValueError('候補は雨・飴の2件')
    for x in [*acoustic, *prior]:
        finite(x, 1e-12, 1, '尤度・事前確率')
    if not math.isclose(sum(prior), 1, rel_tol=1e-09, abs_tol=1e-09):
        raise ValueError('事前確率の合計は1')
    finite(weight, 0, 3, 'λ')
    scores = [math.log(a) + weight * math.log(p) for a, p in zip(acoustic, prior)]
    return {'candidates': ['雨', '飴'], 'scores': scores, 'winner': ['雨', '飴'][scores.index(max(scores))]}


### 計算の定義：`lesson_speech`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_speech(weight=1):
    acoustic,prior = (.6,.4),(.2,.8)
    result = recognize(acoustic,prior,weight)
    rows = [(word,acoustic[j],math.log(acoustic[j]),prior[j],weight*math.log(prior[j]),result['scores'][j]) for j,word in enumerate(result['candidates'])]
    return {'title':'音声候補：音の尤度と言語の事前確率を合わせる','question':'音だけの候補と文脈を使う候補は同じか',
            'input':f'架空の2候補、言語の重みλ={weight}','columns':['候補','音の尤度','log尤度','事前確率','λ×log事前','合計スコア'],'rows':rows,
            'result':f"大きいスコアの候補={result['winner']}",
            'comparison':f"λ=0（音だけ）なら{recognize(weight=0)['winner']}",
            'explanation':'録音を解析した結果ではなく、固定値で候補選択の仕組みを見る教材。スコアは確率そのものではない。',
            'reset':'weight=1で再実行。'}


In [ ]:
example_weight = 1
program_observation_speech = lesson_speech(example_weight)
learning_show(program_observation_speech, reset='example_weight=1に戻して再実行します。')


## 基準結果と既存の変更実験

ここまでの図表を、従来の基準出力と照合します。以下の実験も入力のコピーを使います。定義を変更した場合は再起動して全セル実行してください。


### 計算の定義：`pairs`

中心語と、指定した窓の周辺語を順番に組にする。


In [ ]:
def pairs(corpus=CORPUS, window=1):
    """中心語と、指定した窓の周辺語を順番に組にする。"""
    vocab = validate_corpus(corpus)
    window = integer(window, 1, 3, '窓幅')
    training_pairs = []
    for sentence in corpus:
        for center in range(len(sentence)):
            left = max(0, center-window)
            right = min(len(sentence), center+window+1)
            for neighbor in range(left, right):
                if center != neighbor:
                    training_pairs.append([vocab.index(sentence[center]),vocab.index(sentence[neighbor])])
    return training_pairs


### 計算の定義：`train`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def train(corpus=CORPUS, epochs=20, rate=0.1, seed=65535, window=1):
    vocab = validate_corpus(corpus)
    epochs = integer(epochs, 0, 200, 'エポック')
    finite(rate, 0, 1, '学習率')
    training = pairs(corpus, window)
    if not training:
        raise ValueError('周辺語のある文が必要')
    inputs, outputs = initial_vectors(len(vocab), seed)

    def loss():
        return sum((objective(inputs, outputs, c, t)['loss'] for c, t in training)) / len(training)
    history = [loss()]
    for _ in range(epochs):
        for center, target in training:
            inputs, outputs, _ = update(inputs, outputs, center, target, rate)
        history.append(loss())
    return {'vocab': vocab, 'pairs': training, 'inputs': inputs, 'outputs': outputs, 'loss_history': history}


### 計算の定義：`rounded`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def rounded(value):
    if isinstance(value, float): return round(value, 6)
    if isinstance(value, list): return [rounded(v) for v in value]
    if isinstance(value, dict): return {k: rounded(v) for k, v in value.items()}
    return value


### 計算の定義：`demo`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def demo():
    vectors = representations(); trained = train(); a, b = initial_vectors(5)
    return rounded({'morph': analyze('研究生活'), 'chars': char_split('猫が眠る'),
                    'bigram': bigram(), 'tfidf': tfidf(), 'vectors': vectors,
                    'similarity': cosine(vectors['cooccurrence'][0], vectors['cooccurrence'][3]),
                    'first_update': update(a, b, 0, 1)[2], 'training': trained,
                    'speech': recognize(), 'speech_no_prior': recognize(weight=0)})


### 計算の定義：`experiments`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def experiments():
    lex = copy.deepcopy(LEXICON); lex[-1] = ('活', '名詞', 0)
    docs = copy.deepcopy(CORPUS); docs.append(['猫', 'が', '眠る'])
    return {'dictionary': analyze('研究生活', lex), 'alpha': bigram(alpha=0),
            'document': tfidf(docs), 'window': representations(window=2),
            'learning_rate': train(rate=0), 'language_weight': recognize(weight=0)}


In [ ]:
BOS, EOS = '<s>', '</s>'


## 基準結果

研究／生活の合計2、が→眠る0.3、猫のTF-IDF 0.231049、共起類似度1、λ＝1で飴を確認します。以下の変数を上書きしないでください。


In [ ]:
baseline_result = demo()
print(json.dumps(baseline_result, ensure_ascii=False, indent=2))


## 変更：辞書コスト

問い：活のコストが下がると、分割は変わるか。

予想：研究生／活が選ばれると予想する。

想定結果：研究／生活＝2に対し、研究生／活＝1となる。

説明：長い語を優先したのか、合計コストを比較したのか。

復帰：このセルを元に戻すか、元のNotebookでカーネル再起動後に全セル実行します。


In [ ]:
trial_lexicon = copy.deepcopy(LEXICON)
trial_lexicon[-1] = ('活', '名詞', 0)
trial_dictionary = analyze('研究生活', trial_lexicon)
print(trial_dictionary['tokens'], trial_dictionary['cost'])


## 変更：平滑化α

問い：αを0にすると未観測の組はどうなるか。

予想：が→猫の確率が0になると予想する。

想定結果：が→眠るとが→走るは各0.5。それ以外は0。確率合計は1。

説明：分母が10から4へ変わるのはなぜか。

復帰：このセルを元に戻すか、元のNotebookでカーネル再起動後に全セル実行します。


In [ ]:
trial_alpha = bigram(alpha=0)
print(trial_alpha['probabilities'])


## 変更：文書追加

問い：先頭文書を変更しなくても、そのTF-IDFは変わるか。

予想：猫を含む文書が増えると猫のIDFが下がると予想する。

想定結果：先頭文書の猫のTFは1/3のまま。DF＝3、文書数＝5、TF-IDF≈0.170275。

説明：DFと総出現回数をどう区別したか。

復帰：このセルを元に戻すか、元のNotebookでカーネル再起動後に全セル実行します。


In [ ]:
trial_corpus = copy.deepcopy(CORPUS)
trial_corpus.append(['猫', 'が', '眠る'])
trial_document = tfidf(trial_corpus)
print(trial_document['scores'][0])


## 変更：共起窓幅

問い：左右2語まで数えると猫の行はどう変わるか。

予想：眠る・走るの列が増えると予想する。

想定結果：猫の行は[0,2,1,0,1]。猫と犬の共起類似度はこのコーパスでは1のまま。

説明：行の変化と、類似度の変化は同じものか。

復帰：このセルを元に戻すか、元のNotebookでカーネル再起動後に全セル実行します。


In [ ]:
trial_window = representations(window=2)
print(trial_window['cooccurrence'][0])


## 変更：学習率

問い：学習率0で20エポック処理すると何が変わるか。

予想：重みも損失も変わらないと予想する。

想定結果：入力・出力ベクトルが初期値のままで、平均損失グラフは水平。

説明：回数を増やすことだけで学習が進むと言えるか。

復帰：このセルを元に戻すか、元のNotebookでカーネル再起動後に全セル実行します。


In [ ]:
trial_learning_rate = train(rate=0)
print(trial_learning_rate['loss_history'])


## 変更：言語情報の重み

問い：λ＝0でも候補は飴になるか。

予想：音響尤度が大きい雨になると予想する。

想定結果：候補は雨。λ＝1では飴。順位スコアは正答率ではない。

説明：録音の認識精度が分かったといえないのはなぜか。

復帰：このセルを元に戻すか、元のNotebookでカーネル再起動後に全セル実行します。


In [ ]:
trial_language_weight = recognize(weight=0)
print(trial_language_weight['winner'])


## 基準を保持したか

別変数・コピーを使ったため、基準定数と結果は変わりません。エラー時はカーネル選択、全セル実行、入力範囲を確認してください。


In [ ]:
assert baseline_result == demo()
assert CORPUS == [['猫','が','眠る'],['犬','が','眠る'],['猫','が','走る'],['犬','が','走る']]
print('基準入力と結果を保持しています。')


## 振り返りと復帰

入力・行った処理・結果・変更前との差を、自分の言葉で説明してください。観察セルは毎回基準のコピーから計算するため、同じセルを再実行しても前の実験の続きにはなりません。NameErrorや実行順が不明な場合はカーネル再起動→全セル実行。元の配布ファイルを残し、変更版は別名保存します。

Python版は`python natural_language_processing.py --explain all`で同じ計算表を表示します。各テーマは`--explain morph`のように指定できます。従来のコマンドと固定期待値も維持しています。
